In [1]:
"""

LICENSE MIT
2020
Guillaume Rozier
Website : http://www.covidtracker.fr
Mail : guillaume.rozier@telecomnancy.net

README:
This file contains scripts that download data from data.gouv.fr and then process it to build many graphes.

The charts are exported to 'charts/images/france'.
Data is download to/imported from 'data/france'.
Requirements: please see the imports below (use pip3 to install them).

"""

"\n\nLICENSE MIT\n2020\nGuillaume Rozier\nWebsite : http://www.covidtracker.fr\nMail : guillaume.rozier@telecomnancy.net\n\nREADME:\nThis file contains scripts that download data from data.gouv.fr and then process it to build many graphes.\n\nThe charts are exported to 'charts/images/france'.\nData is download to/imported from 'data/france'.\nRequirements: please see the imports below (use pip3 to install them).\n\n"

In [98]:
import pandas as pd
import france_data_management as data
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import plotly
PATH = "../../"
import locale
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')

'fr_FR.UTF-8'

In [102]:
data.download_data()

36it [00:07,  4.61it/s]                      


In [103]:
df_hosp = data.import_data_hosp_clage().groupby(["jour", "cl_age90"]).sum().reset_index()
df_hosp_nouveaux = data.import_data_new().groupby("jour").sum().reset_index()

df_hosp = df_hosp[df_hosp.cl_age90 == 0].reset_index()

df_tests_viro = data.import_data_tests_sexe()
df_tests_viro = df_tests_viro[df_tests_viro.cl_age90 == 0].reset_index()

In [104]:
df = df_tests_viro.merge(df_hosp_nouveaux, left_on="jour", right_on="jour")

df = df.reset_index()
df["hosp_cas_ratio"] = df.incid_hosp.rolling(window=7).mean()/df.P.rolling(window=7).mean().shift(7) * 100
df = df[df["jour"] >= "2021-01-01"]

In [105]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.jour,
    y=df.hosp_cas_ratio))
fig.update_yaxes(ticksuffix=" %")
fig.update_layout(
    title={
                        'text': "Proportion des cas qui sont hospitalisés",
                        'y':0.97,
                        'x':0.5,
                        'xanchor': 'center',
                        'yanchor': 'top'},
    titlefont = dict(
                    size=30),
    annotations = [
                        dict(
                            x=0.5,
                            y=1.12,
                            xref='paper',
                            yref='paper',
                            font=dict(size=14),
                            text="Proportion des admissions à l'hôpital par rapport aux cas positifs 7 jours plus tôt<br>{} - @GuillaumeRozier - covidtracker.fr".format(datetime.strptime(df.jour.max(), '%Y-%m-%d').strftime('%d %B %Y')),#'Date : {}. Source : Santé publique France. Auteur : GRZ - covidtracker.fr.'.format(),                    showarrow = False
                            showarrow=False
                        ),
                        ]
)

name_fig = "cas_hospitalisations_ratio"
fig.write_image(PATH + "images/charts/france/{}.jpeg".format(name_fig), scale=2, width=900, height=600)
plotly.offline.plot(fig, filename = PATH + 'images/html_exports/france/{}.html'.format(name_fig), auto_open=False)
        

'../../images/html_exports/france/cas_hospitalisations_ratio.html'

In [106]:
"""fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=df.jour,
    y=df.hosp,
    name="Hosp 70-79",),
    secondary_y=False)
fig.add_trace(go.Scatter(
    x=df.jour,
    y=df.P.rolling(window=7).mean(),
    name="cas positifs 70-79",),
    secondary_y=True)"""

'fig = make_subplots(specs=[[{"secondary_y": True}]])\nfig.add_trace(go.Scatter(\n    x=df.jour,\n    y=df.hosp,\n    name="Hosp 70-79",),\n    secondary_y=False)\nfig.add_trace(go.Scatter(\n    x=df.jour,\n    y=df.P.rolling(window=7).mean(),\n    name="cas positifs 70-79",),\n    secondary_y=True)'